# Download original CAL-028 and CAL-015 trajectories from RunPod S3

This notebook **copies** your original production DCD segments and AMBER topology files from RunPod network volume `cal028-md` to **Google Drive**. It never changes or deletes objects on RunPod. Keep your paid GPU stopped.

- **Source bucket:** `lf87cdnmsz` (`EU-RO-1`)
- **Output:** `MyDrive/CYP51_MD_animation_sources/CAL028` and `/CAL015`
- **Expected:** 50 segments for each compound, each approximately 86 MB (~8.6 GB combined), plus topologies. Make sure Google Drive has enough free space.
- Run cells in order on **Google Colab CPU**. Credentials are prompted privately, **not saved** to Drive or to this notebook. Do not share credentials.
- The download checks the file sizes and **skips completed files** when resumed after an interruption. A partially written file is replaced safely on a retry.

In [ ]:
!pip -q install boto3

from getpass import getpass
import boto3, os, re, json, shutil, time
from pathlib import Path
from datetime import datetime, timezone
from botocore.config import Config
from boto3.s3.transfer import TransferConfig

S3_BUCKET = 'lf87cdnmsz'
S3_ENDPOINT = 'https://s3api-eu-ro-1.runpod.io'
S3_REGION = 'EU-RO-1'

# A new Colab runtime needs a fresh private login; never paste the secret into a code cell.
access_key = getpass('RunPod S3 access key / user ID: ')
secret_key = getpass('RunPod S3 secret access key: ')
s3 = boto3.client(
    's3', endpoint_url=S3_ENDPOINT, region_name=S3_REGION,
    aws_access_key_id=access_key, aws_secret_access_key=secret_key,
    config=Config(signature_version='s3v4', s3={'addressing_style':'path'},
                  retries={'max_attempts': 6, 'mode':'standard'})
)
# Credential strings are no longer needed after the client has been constructed.
del access_key, secret_key
print('Connected. Credentials not printed or saved.')

In [ ]:
# Discover and validate source objects before downloading.
PREFIXES = {
    'CAL028': 'CAL028_5090_results/',
    'CAL015': 'CAL-015/production_50ns_1fs/',
}
KNOWN_TOP = {
    'CAL028': 'cal028.prmtop',
    'CAL015': 'CAL-015/N14_frame_corrected/system.prmtop',
}

paginator = s3.get_paginator('list_objects_v2')

def list_objects(prefix):
    return [obj for pg in paginator.paginate(Bucket=S3_BUCKET, Prefix=prefix)
            for obj in pg.get('Contents', [])]

def find_top(system):
    preferred = KNOWN_TOP[system]
    try:
        return preferred, int(s3.head_object(Bucket=S3_BUCKET, Key=preferred)['ContentLength'])
    except Exception:
        candidates = [o for o in list_objects('') if o['Key'].lower().endswith('.prmtop')]
        print('Available .prmtop files:', [o['Key'] for o in candidates])
        raise RuntimeError(f'Expected topology not found for {system}: {preferred}. '
                           'Do not download trajectories without verifying the topology.')

sources = {}
for system, prefix in PREFIXES.items():
    objlist = list_objects(prefix)
    pattern = re.compile(r'^' + re.escape(prefix) + r'ns_(\d{3})/trajectory\.dcd$')
    segs = {}
    for obj in objlist:
        m = pattern.fullmatch(obj['Key'])
        if m:
            segs[int(m.group(1))] = {'key': obj['Key'], 'size': int(obj['Size'])}
    missing = sorted(set(range(1, 51)) - set(segs))
    extra = sorted(set(segs) - set(range(1, 51)))
    print(f'{system}: {len(segs)} DCD segments; missing={missing}, unexpected={extra}')
    if missing or extra:
        raise RuntimeError(f'{system}: Expected precisely ns_001..ns_050; inspect source objects first.')
    top_key, top_size = find_top(system)
    print('  Topology:', top_key, f'({top_size / 1024**2:.1f} MiB)')
    print('  DCD size:', f'{sum(x["size"] for x in segs.values()) / 1024**3:.2f} GiB')
    sources[system] = {'topology': {'key': top_key, 'size': top_size}, 'segments': segs}
print('Total expected transfer:', f'{sum(sum(v["size"] for v in x["segments"].values())+x["topology"]["size"] for x in sources.values()) / 1024**3:.2f} GiB')

In [ ]:
# Destination: permanent Google Drive storage (not the temporary /content runtime).
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_ROOT = Path('/content/drive/MyDrive/CYP51_MD_animation_sources')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TEMP_ROOT = Path('/content/_runpod_transfer_temp')
TEMP_ROOT.mkdir(parents=True, exist_ok=True)
print('Output:', OUTPUT_ROOT)
print('Free temporary Colab space:', f'{shutil.disk_usage("/content").free / 1024**3:.1f} GiB')
print('Confirm Google Drive has ~9 GiB available before downloading.')

In [ ]:
# Download original topology + DCD files, one at a time, with file-size checks.
# Re-run this cell any time: it skips matching existing files and retries missing/incomplete ones.
transfer_config = TransferConfig(multipart_threshold=1024**3, use_threads=False)

def save_object(source, destination):
    key, expected_size = source['key'], source['size']
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size == expected_size:
        return 'skipped'
    stage = TEMP_ROOT / 'current_download.partial'
    stage.unlink(missing_ok=True)
    dest_tmp = destination.with_name(destination.name + '.partial')
    dest_tmp.unlink(missing_ok=True)
    try:
        s3.download_file(S3_BUCKET, key, str(stage), Config=transfer_config)
        if stage.stat().st_size != expected_size:
            raise IOError(f'Download size mismatch for {key}')
        shutil.copyfile(stage, dest_tmp)
        if dest_tmp.stat().st_size != expected_size:
            raise IOError(f'Drive copy size mismatch for {key}')
        os.replace(dest_tmp, destination)
        return 'downloaded'
    finally:
        stage.unlink(missing_ok=True)

for system, data in sources.items():
    d = OUTPUT_ROOT / system
    print(f'\n=== {system} ===')
    status = save_object(data['topology'], d / 'system.prmtop')
    print('Topology:', status)
    for ns, src in sorted(data['segments'].items()):
        fname = f'ns_{ns:03d}.dcd'
        status = save_object(src, d / fname)
        print(f'  {fname} : {status}', flush=True)

manifest = {
    'source_bucket': S3_BUCKET,
    'source_endpoint': S3_ENDPOINT,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'systems': {
        name: {
            'topology': data['topology'],
            'segments': {f'{i:03d}': val for i, val in data['segments'].items()}
        } for name, data in sources.items()
    },
}
(OUTPUT_ROOT / 'source_manifest.json').write_text(json.dumps(manifest, indent=2))
print('Done. Manifest saved:', OUTPUT_ROOT / 'source_manifest.json')

In [ ]:
# Final audit (all 50 segments, topology, and exact recorded file sizes for each compound).
for system, data in sources.items():
    d = OUTPUT_ROOT / system
    required = [(d / 'system.prmtop', data['topology']['size'])] + [
        (d / f'ns_{i:03d}.dcd', data['segments'][i]['size']) for i in range(1, 51)
    ]
    bad = [(str(p), expected) for p, expected in required
           if not p.is_file() or p.stat().st_size != expected]
    if bad:
        print(f'{system}: INCOMPLETE or mismatched:', bad[:10])
    else:
        size = sum(sz for _, sz in required) / 1024**3
        print(f'{system}: VERIFIED — 50 DCDs + topology ({size:.2f} GiB)')
print('Stored at:', OUTPUT_ROOT)